# Music-to-Dance Generation via Atomic Movements — Google Colab

**学習は不要です。** 配布されている学習済みチェックポイントを使えば、
セクション 1〜5 を上から流すだけでダンス生成まで行けます（GPU で十数分程度、
大半はデータセットのダウンロード時間です）。

自分で学習し直したい場合だけ、任意のセクション 6 を使ってください（数時間かかります）。

| セクション | 内容 | 所要時間の目安 |
| --- | --- | --- |
| 1〜2 | 環境構築・データセット取得 | 10 分前後 |
| 3〜5 | 学習済みモデルの取得 → 生成 → 描画 | 数分 |
| 6（任意） | 自分で学習する | 数時間〜 |
| 7（任意） | 評価（AIST++ と SMPL が別途必要） | 数十分 |

**実行前に:** *ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ → GPU*
を選んでください（T4 で足ります）。

## 1. 環境構築

どのコードが動くかを決めるのは、このノートブックをどこから開いたかではなく
**下の clone セル**です。ブランチからこのノートブックを開いた場合は、`BRANCH` に
同じブランチ名を入れてください。

すでに `/content/AtomicDance` がある場合も、このセルは指定ブランチに強制的に
切り替えます（ローカルの変更は破棄されます）。

**再実行するときの注意:** clone セルはリポジトリのコードを最新にしますが、
**ブラウザで開いているこのノートブックのセル自体は更新されません**。セルの内容も
新しくしたい場合は、GitHub からノートブックを開き直してください
（*ファイル → ノートブックを開く → GitHub*）。下の確認セルは、チェックアウトが
origin より古ければその場で止めます。

In [ ]:
!nvidia-smi || echo "GPU が見つかりません。ランタイム > ランタイムのタイプを変更 > GPU を選んでください"

In [ ]:
import os

REPO_URL = "https://github.com/yamak493/AtomicDance.git"
REPO_DIR = "/content/AtomicDance"
# Colab 対応が main にマージされたら "main" に変更してください。
BRANCH = "claude/laughing-bardeen-73mdcu"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone --branch "{BRANCH}" "{REPO_URL}" $REPO_DIR
else:
    # 既存のクローンを指定ブランチに合わせ直す（BRANCH を変えて再実行したとき用）
    !git -C $REPO_DIR fetch origin "{BRANCH}"
    !git -C $REPO_DIR checkout -B "{BRANCH}" "origin/{BRANCH}"

%cd $REPO_DIR
!git log --oneline -1

チェックアウトしたのが Colab 対応版かどうかを確認します。ここで失敗する場合は
`BRANCH` の指定が違います。

In [ ]:
import subprocess
from pathlib import Path

REQUIRED = ("compat/rotation_conversions.py", "compat/smpl.py",
            "tests/__init__.py", "requirements-colab.txt")
missing = [name for name in REQUIRED if not Path(name).exists()]
if missing:
    raise RuntimeError(
        "Colab 対応版のファイルが見つかりません: {}\n"
        "上の clone セルの BRANCH を Colab 対応が入ったブランチに変えて実行し直してください。"
        .format(", ".join(missing))
    )


def git(*arguments):
    return subprocess.run(["git", "-C", REPO_DIR] + list(arguments),
                          capture_output=True, text=True).stdout.strip()


# チェックアウトが origin より古いと、修正済みのはずのバグを踏み続けることになる
local = git("rev-parse", "HEAD")
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin", BRANCH],
               capture_output=True)
remote = git("rev-parse", "FETCH_HEAD")
if remote and local != remote:
    raise RuntimeError(
        "チェックアウトが古いです。\n"
        "  手元 : {} {}\n"
        "  origin: {} {}\n"
        "上の clone セルを実行し直してください。\n"
        "さらに、ノートブックのセル自体は git では更新されません。セルの内容も新しく"
        "したい場合は、GitHub からノートブックを開き直してください。"
        .format(local[:9], git("log", "-1", "--format=%s", "HEAD"),
                remote[:9], git("log", "-1", "--format=%s", "FETCH_HEAD"))
    )

print("Colab 対応版の最新をチェックアウトしています:", local[:9], git("log", "-1", "--format=%s"))

Colab には CUDA 版の PyTorch が最初から入っているので、`requirements-colab.txt` は
その周辺で足りないものだけを入れます。リポジトリの `compat/` パッケージが PyTorch3D
（Colab 用のホイールが無く、ソースビルドに数十分かかる）を純 PyTorch 実装で置き換える
ので、ここでコンパイルされるものはありません。

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA 利用可否:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("デバイス:", torch.cuda.get_device_name(0))

出力先の設定です。**自分で学習する場合（セクション 6）だけ** `USE_DRIVE = True` に
してください。Drive に置いておけば、セッションが切れてもチェックポイントが残り、
`--resume` で続きから流せます。生成するだけなら False のままで構いません
（Drive のマウント認証が不要になります）。

In [ ]:
from pathlib import Path

USE_DRIVE = False  # 自分で学習するなら True

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/AtomicDance")
else:
    WORK_DIR = Path("/content/atomicdance-work")

RUNS_DIR = WORK_DIR / "runs"
OUTPUT_DIR = WORK_DIR / "outputs"
for directory in (RUNS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("チェックポイント ->", RUNS_DIR)
print("出力             ->", OUTPUT_DIR)

（任意）コードがこのランタイムで動くことをテストで確認します。20 秒ほどです。

In [ ]:
!python -m unittest discover -s tests -t .

## 2. atomic データセットの取得

処理済みの `atomic_aistpp` パッケージには、フレーム同期済みのモーション、35 次元の
音楽特徴、atomic ラベルが入っています。ラベル `1..100` が動きのカテゴリ、`0` が
トランジションです。

**生成するだけでもこれは必要です。** 推論時に、atomic ラベルに対応する動きの
プロトタイプをこの学習データから検索して下書きを組み立てるためです。

下のセルはダウンロード・展開・配置を検証付きで行い、アーカイブ内の構成が
どうなっていても `data/atomic_aistpp/` に揃えます。

In [ ]:
import shutil
import zipfile
from pathlib import Path

DATASET_URL = "https://drive.google.com/file/d/1ETsaetMMWeKV3_E3Lr40BdybAsUAG8WM/view"
ARCHIVE = Path("/content/atomic_aistpp.zip")
STAGING = Path("/content/atomic_aistpp_extracted")
TARGET = Path("data/atomic_aistpp")

if (TARGET / "train" / "motion.npy").exists():
    print("配置済み:", TARGET.resolve())
else:
    if not ARCHIVE.exists():
        !pip install -q --upgrade gdown
        !gdown --fuzzy "{DATASET_URL}" -O "{ARCHIVE}"

    if not ARCHIVE.exists():
        raise RuntimeError(
            "ダウンロードできませんでした。下の「手動ダウンロード」の手順を使ってください。"
        )
    if not zipfile.is_zipfile(ARCHIVE):
        preview = ARCHIVE.read_bytes()[:300]
        ARCHIVE.unlink()
        raise RuntimeError(
            "ZIP ではないファイルが落ちてきました（Drive のダウンロード上限が原因のことが"
            "多いです）。下の「手動ダウンロード」を使ってください。\n先頭バイト: {!r}"
            .format(preview)
        )

    shutil.rmtree(STAGING, ignore_errors=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall(STAGING)

    # アーカイブ内のどの階層に train/ があっても拾う
    found = sorted(STAGING.rglob("train/motion.npy"))
    if not found:
        listing = sorted(str(path.relative_to(STAGING)) for path in STAGING.rglob("*"))
        raise RuntimeError(
            "展開結果に train/motion.npy がありません。中身:\n  "
            + "\n  ".join(listing[:40])
        )
    source = found[0].parent.parent

    TARGET.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(TARGET, ignore_errors=True)
    shutil.move(str(source), str(TARGET))
    print("配置しました:", TARGET.resolve())

print(sorted(path.name for path in TARGET.iterdir()))

**手動ダウンロード（`gdown` が失敗したとき）**

1. [データセットのリンク](https://drive.google.com/file/d/1ETsaetMMWeKV3_E3Lr40BdybAsUAG8WM/view?usp=sharing)
   をブラウザで開いてダウンロードする
2. Colab 左の**ファイル**ペインに ZIP をドラッグして `/content/atomic_aistpp.zip`
   として置く（あるいは自分の Drive に入れて `/content/drive/MyDrive/...` からコピー）
3. 上のセルを再実行する（ZIP が既にあればダウンロードは飛ばします）

## 3. 学習済みチェックポイントの取得

ここが通れば学習は不要です。配布フォルダから planner と completion の
チェックポイントを落として、`RUNS_DIR` 以下に配置します。ファイル名ではなく
チェックポイント内の `stage` フィールドで振り分けるので、命名が違っていても
正しい場所に入ります。

すでに `RUNS_DIR` にチェックポイントがある場合（セクション 6 で学習した場合も含む）
はダウンロードを飛ばし、最新のものを使います。

In [ ]:
import shutil
from pathlib import Path

from compat import torch_load

CHECKPOINTS_URL = "https://drive.google.com/drive/folders/1r707t1FKhs_FkHNkNbqtDxIaXiYMUZuq"
DOWNLOAD_DIR = Path("/content/atomicdance_checkpoints")

PLANNER_DIR = RUNS_DIR / "atomic_planner"
COMPLETION_DIR = RUNS_DIR / "atomic_completion"


def newest_checkpoint(directory):
    directory = Path(directory)
    if not directory.is_dir():
        return None
    checkpoints = sorted(directory.glob("*.pt"), key=lambda path: path.stat().st_mtime)
    return checkpoints[-1] if checkpoints else None


if newest_checkpoint(PLANNER_DIR) is None or newest_checkpoint(COMPLETION_DIR) is None:
    !pip install -q --upgrade gdown
    !gdown --folder "{CHECKPOINTS_URL}" -O "{DOWNLOAD_DIR}" --remaining-ok

    moved = 0
    for path in sorted(DOWNLOAD_DIR.rglob("*.pt")) if DOWNLOAD_DIR.is_dir() else []:
        try:
            stage = torch_load(path, mmap=True).get("stage")
        except Exception as error:
            print("読めませんでした:", path.name, "->", error)
            continue
        if stage == "planner":
            destination = PLANNER_DIR
        elif stage == "completion":
            destination = COMPLETION_DIR
        else:
            print("stage 不明のためスキップ:", path.name, "(stage={!r})".format(stage))
            continue
        destination.mkdir(parents=True, exist_ok=True)
        shutil.move(str(path), str(destination / path.name))
        moved += 1
    print("配置したチェックポイント:", moved)

PLANNER_CHECKPOINT = newest_checkpoint(PLANNER_DIR)
COMPLETION_CHECKPOINT = newest_checkpoint(COMPLETION_DIR)

if PLANNER_CHECKPOINT is None or COMPLETION_CHECKPOINT is None:
    raise RuntimeError(
        "学習済みチェックポイントを取得できませんでした。\n"
        "  planner   : {}\n"
        "  completion: {}\n"
        "配布フォルダがまだ公開されていない可能性があります。ブラウザで中身を確認し\n"
        "  {}\n"
        "手動で {} と {} に .pt を置いて、このセルを再実行してください。\n"
        "公開されていない場合は、セクション 6 で自分で学習する必要があります。"
        .format(PLANNER_CHECKPOINT, COMPLETION_CHECKPOINT,
                CHECKPOINTS_URL, PLANNER_DIR, COMPLETION_DIR)
    )

print("planner   :", PLANNER_CHECKPOINT)
print("completion:", COMPLETION_CHECKPOINT)

## 4. ダンスの生成

`--audio-dir` に WAV の入ったフォルダを指定します。atomic データセットには音声が
含まれないので、下のセルで自分の曲をアップロードするのが手軽です。AIST++ の WAV を
`data/edge_aistpp/wavs` に置く手もあります（セクション 7 参照）。

AIST++ の命名（`gWA_sBM_c01_d25_mWA4_ch05.wav`）だとファイル名からテンポを読み取ります。
それ以外の名前ならビートトラッキングにフォールバックするので、どちらでも問題ありません。

In [ ]:
# 自分の .wav をアップロードして生成に使う
from pathlib import Path

from google.colab import files

CUSTOM_AUDIO_DIR = Path("/content/custom_music")
CUSTOM_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
for filename, content in files.upload().items():
    (CUSTOM_AUDIO_DIR / filename).write_bytes(content)
print(sorted(path.name for path in CUSTOM_AUDIO_DIR.glob("*.wav")))

`--max-samples 3` で最初の 3 曲だけ処理します。フォルダ全体を生成するときは
この行を外してください。`--max-frames 150` は 30 FPS で 5 秒分です。

In [ ]:
AUDIO_DIR = str(CUSTOM_AUDIO_DIR)  # AIST++ を使うなら "data/edge_aistpp/wavs"
GENERATED_DIR = OUTPUT_DIR / "generated"

!python infer_atomic.py \
  --audio-dir "{AUDIO_DIR}" \
  --output-dir "{GENERATED_DIR}" \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --data-root data/atomic_aistpp \
  --device cuda \
  --max-frames 150 \
  --max-samples 3 \
  --inference-batch-size 4

## 5. 結果を描画する

`skeleton_render` はスケルトンの GIF を書き出し、音声と合わせて MP4 にします
（ffmpeg は Colab に入っています）。描画は CPU 処理なので、試行中はクリップを
短くしておくと快適です。

In [ ]:
import pickle
from pathlib import Path

import numpy as np

from vis import skeleton_render

generated = sorted(Path(GENERATED_DIR).glob("*.pkl"))
print("生成されたモーション:", len(generated))

motion_path = generated[0]
with open(motion_path, "rb") as handle:
    data = pickle.load(handle)

audio_path = Path(data["audio_path"])
if not audio_path.is_file():
    raise FileNotFoundError(
        "推論時に使った音声が見つかりません: {}\n"
        "VM が初期化されると /content 以下は消えます。音声を置き直してセクション 4 の"
        "推論からやり直してください。".format(audio_path)
    )

frames = 150  # 30 FPS で 5 秒
render_dir = OUTPUT_DIR / "renders"
skeleton_render(
    np.asarray(data["full_pose"])[:frames],
    epoch="atomic",
    out=str(render_dir),
    name=str(audio_path),
    sound=True,
    contact=np.asarray(data["contacts"])[:frames],
)

# skeleton_render の出力名は "{epoch}_{音声のファイル名}.mp4"
VIDEO_PATH = render_dir / "atomic_{}.mp4".format(audio_path.stem)
print("書き出しました:", VIDEO_PATH, "({:,} バイト)".format(VIDEO_PATH.stat().st_size))

In [ ]:
import base64

from IPython.display import HTML

encoded = base64.b64encode(VIDEO_PATH.read_bytes()).decode()
HTML('<video width=480 controls><source src="data:video/mp4;base64,{}" type="video/mp4"></video>'
     .format(encoded))

## 6.（任意）自分で学習する

**生成するだけならこのセクションは不要です。** セクション 3 の学習済み
チェックポイントで足ります。

論文の設定で回すと planner が 20 エポック、completion が 200 エポックで、合計
数時間〜になります。無料枠の Colab セッションより長いので、必ず上の設定セルで
`USE_DRIVE = True` にしてから始めてください。チェックポイントが Drive に残り、
切断後に `--resume` で再開できます。

Colab の VM は CPU 2 コアなので、既定の 4 より `--workers 2` のほうが速く回ります。
下のバッチサイズは 16 GB の T4 に収まる値です。メモリ不足が出たら下げてください。

まずスモークテストで、学習ループ自体が数秒で通ることを確認します。

In [ ]:
!python train_atomic.py \
  --stage planner \
  --data-root data/atomic_aistpp \
  --output-dir /tmp/smoke_planner \
  --device cuda \
  --epochs 1 --max-steps 2 --batch-size 2 --workers 2

### Atomic movement planner

In [ ]:
!python train_atomic.py \
  --stage planner \
  --data-root data/atomic_aistpp \
  --output-dir "{RUNS_DIR}/atomic_planner" \
  --device cuda \
  --epochs 20 \
  --batch-size 16 \
  --workers 2

### Dance completion model

チェックポイントは 20 エポックごとに保存されます。セッションが切れたら、
`RUNS_DIR/atomic_completion` の最新ファイルを `--resume` に指定して再実行してください。

In [ ]:
!python train_atomic.py \
  --stage completion \
  --data-root data/atomic_aistpp \
  --output-dir "{RUNS_DIR}/atomic_completion" \
  --device cuda \
  --epochs 200 \
  --batch-size 8 \
  --workers 2

学習したチェックポイントを使うには、セクション 3 のセルを再実行してください。
`RUNS_DIR` にあるものを検出して `PLANNER_CHECKPOINT` / `COMPLETION_CHECKPOINT` を
最新に更新します（ダウンロードは行いません）。そのあとセクション 4 に戻ります。

## 7.（任意）評価

評価指標は生成モーションを AIST++ の正解と比較するので、このセクションだけは
プロジェクトの配布物に含まれない、それぞれ別ライセンスの素材が 2 つ必要です。

- AIST++ のモーション PKL と WAV を `data/edge_aistpp/{motions,wavs}` に配置
  （[AIST++ 公式サイト](https://google.github.io/aistplusplus_dataset/)）
- `SMPL_MALE.pkl` を `smpl/SMPL_MALE.pkl` に配置
  （[SMPL 公式サイト](https://smpl.is.tue.mpg.de/)）

SMPL の配布ファイルは配列を chumpy オブジェクトとして保存していますが、chumpy は
Colab の Python では import できません。`compat/smpl.py` が chumpy 無しで読むので、
ライセンス取得した `.pkl` をそのまま置けば動きます。

評価器は kinetic / manual 特徴の FID と diversity、そして Beat Alignment Score を
出します。キャッシュを使わず作り直すには `--overwrite-inference --force-extract`
を足してください。

In [ ]:
!python -m eval.evaluate \
  --ground-truth-motions data/edge_aistpp/motions \
  --audio-dir data/edge_aistpp/wavs \
  --sequence-list data/splits/crossmodal_test.txt \
  --plan-source planner \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --atomic-data-root data/atomic_aistpp \
  --smpl-model smpl/SMPL_MALE.pkl \
  --device cuda:0 \
  --max-inference-frames 150 \
  --inference-batch-size 4 \
  --workers 2 \
  --inference-output "{OUTPUT_DIR}/eval_generated" \
  --cache-dir "{OUTPUT_DIR}/eval_cache" \
  --output "{OUTPUT_DIR}/results_planner.json"

In [ ]:
import json

print(json.dumps(json.loads((OUTPUT_DIR / "results_planner.json").read_text()), indent=2))

## トラブルシューティング

**学習済みチェックポイントが落ちてこない** — 配布フォルダがまだ公開されていない
可能性があります。README の
[Checkpoints](https://drive.google.com/drive/folders/1r707t1FKhs_FkHNkNbqtDxIaXiYMUZuq?usp=sharing)
をブラウザで開いて中身を確認してください。空ならセクション 6 で学習するしか
ありません。ファイルがあるのに `gdown` が失敗する場合は、手動でダウンロードして
`RUNS_DIR/atomic_planner/` と `RUNS_DIR/atomic_completion/` に置き、セクション 3 の
セルを再実行してください。

**`ImportError: Start directory is not importable: '.../tests'`** —
`main` をチェックアウトしています。`main` には `tests/__init__.py` が無く、
Python 3.11 以降の unittest は名前空間パッケージを探索できません。clone セルの
`BRANCH` を Colab 対応ブランチにして実行し直してください。

**`FileNotFoundError: data/atomic_aistpp/train/motion.npy`** —
データセットの配置が済んでいません。セクション 2 のセルを実行してください。
`gdown` が Drive のダウンロード上限に当たった場合は、そこに書いてある手動
ダウンロードの手順を使ってください。

**`ModuleNotFoundError: No module named 'compat'`** —
リポジトリのルートから実行していません。`%cd /content/AtomicDance` を実行してください。

**`CUDA out of memory`** — `--batch-size`（学習）や `--inference-batch-size` を
下げて、ランタイムを再起動してアロケータのキャッシュを解放してください。

**学習中にセッションが切れた** — 環境構築のセルを流し直し、`train_atomic.py` に
`--resume <チェックポイント>` を渡します。設定セルで `USE_DRIVE = True` にして
いた場合のみ可能です。

**MP4 が出力されない / `IndexError: list index out of range`** —
`skeleton_render` は ffmpeg を呼びますが、以前は終了ステータスを見ておらず、失敗しても
無言で MP4 が作られませんでした。空白や括弧を含むファイル名（アップロードした曲では
よくあります）でシェルのコマンドが壊れるのが主な原因です。現在は引数をリストで渡すよう
修正済みで、失敗すれば ffmpeg のエラー内容付きで例外になります。古いチェックアウトの
場合は clone セルから流し直してください。

## 既知の差分: 論文環境との音楽特徴のずれ

学習は配布された `music.npy` の特徴を読みますが、推論は音声から特徴を計算し直します。
Colab の librosa（0.11）は固定版（0.9.2）と出力が完全には一致しないため、学習時と
推論時にわずかなずれが生じます。同一音声での実測値:

| 特徴ブロック | 次元 | 差分 |
| --- | --- | --- |
| onset envelope | 1 | なし（ビット一致） |
| MFCC | 20 | なし（ビット一致） |
| chroma CENS | 12 | 絶対 2.1e-2 / 相対 2.5%（最大） |
| onset peak one-hot | 1 | なし |
| beat one-hot | 1 | 241 フレーム中 3 フレーム |

chroma の差は `librosa.cqt` の `res_type` 既定値が 0.10 で `None` → `soxr_hq` に
変わったことによるものです。`None` が選んでいた `auto_resample` 経路は 0.11 では
削除されているため、パラメータ指定で元の挙動には戻せません。beat の差は
`beat_track` の内部変更によるもので、入力の onset envelope は一致しており開始 BPM も
固定して比較しています。

これが生成結果をどの程度動かすかは未計測です。論文の数値を厳密に再現したい場合は、
固定環境（Python 3.7 / librosa 0.9.2）を使ってください。